In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Параметры

In [ ]:
# Параметры одного поезда (для всех одинаковые)
a_acc          = 0.03    # ускорение при разгоне, м/с²
a_brake        = -0.1    # ускорение при торможении, м/с²
m              = 6300    # масса поезда, т
L_train        = 1000    # длина поезда, м
ds             = 50      # шаг по пути, м
safety_margin  = 100     # запас к тормозному пути, м

# Параметр выбега
COAST_MIN      = 45      # минимальная длительность выбега, с (нужно от 30 до 60)

# Маршрут: (s_начало, s_конец, V_max км/ч)
speed_limits_kmh = [
    (0,     5000,   80),
    (5000,  8000,   40),
    (8000,  18000,  80),
    (18000, 22000,  60),
    (22000, 30000,  80),
]
S_stop = 30000           # конец маршрута, м

# Параметры группы
N_TRAINS         = 3     # сколько поездов
HEADWAY_SEC      = 600   # интервал между отправлениями, с
MIN_DIST_TO_LEAD = 800   # минимальная дистанция до хвоста переднего поезда, м

# Переводим ограничения в м/с
speed_limits_ms = [(s1, s2, v / 3.6) for s1, s2, v in speed_limits_kmh]

print(f'Маршрут: {S_stop/1000:.0f} км, длина поезда {L_train} м, шаг {ds} м')
print(f'Поездов в группе: {N_TRAINS}, интервал {HEADWAY_SEC/60:.0f} мин')
print(f'Минимальная дистанция: {MIN_DIST_TO_LEAD} м')
print(f'Минимальный выбег: {COAST_MIN} с')
print()
print('Ограничения скорости:')
for s1, s2, v in speed_limits_kmh:
    print(f'  {s1/1000:5.1f} – {s2/1000:5.1f} км:  {v} км/ч')

## 2. Вспомогательные функции

Считаем тормозной путь, находим ограничения скорости и положение переднего поезда.

In [ ]:
def braking_distance_to_speed(v_cur, v_tgt, a_br):
    """Тормозной путь, чтобы снизить скорость с v_cur до v_tgt (м)."""
    if v_cur <= v_tgt:
        return 0.0
    return (v_cur**2 - v_tgt**2) / (2 * abs(a_br))


def get_speed_limit(s, limits):
    """Ограничение скорости в точке s."""
    for s1, s2, v_lim in limits:
        if s1 <= s < s2:
            return v_lim
    return 0.0


def get_speed_limit_for_tail(s_tail, limits):
    """Ограничение для хвоста. Если хвост ещё не выехал — не учитываем."""
    if s_tail < 0:
        return 999.0
    for s1, s2, v_lim in limits:
        if s1 <= s_tail < s2:
            return v_lim
    return 999.0


def find_next_static_restriction(s_head, v_cur, limits, s_stop):
    """Ближайшая впереди точка снижения скорости или остановки.
    Возвращает расстояние до неё и целевую скорость."""
    best_dist = s_stop - s_head
    best_v    = 0.0     # на конце маршрута — полная остановка
    for s1, s2, v_lim in limits:
        if s1 > s_head and v_lim < v_cur:
            d = s1 - s_head
            if d < best_dist:
                best_dist = d
                best_v    = v_lim
    return best_dist, best_v


def speed_after_step(v, a, ds_step):
    """Скорость после шага ds с ускорением a. Из кинематики v² = v0² + 2·a·s."""
    v2 = v**2 + 2 * a * ds_step
    return np.sqrt(v2) if v2 > 0 else 0.0


class LeadingTrajectory:
    """Хранит траекторию переднего поезда.
    По времени t возвращает положение головы, хвоста и скорость."""
    def __init__(self, df):
        self.t   = df['t_абс_с'].to_numpy()
        self.s_h = df['s_голова_м'].to_numpy()
        self.s_t = df['s_хвост_м'].to_numpy()
        self.v   = df['v_м/с'].to_numpy()
        self.t_min = self.t[0]
        self.t_max = self.t[-1]
        self.s_h_final = self.s_h[-1]
        self.s_t_final = self.s_t[-1]

    def at(self, t):
        if t <= self.t_min:
            return self.s_h[0], self.s_t[0], 0.0
        if t >= self.t_max:
            # передний поезд уже остановился
            return self.s_h_final, self.s_t_final, 0.0
        sh = float(np.interp(t, self.t, self.s_h))
        st = float(np.interp(t, self.t, self.s_t))
        v  = float(np.interp(t, self.t, self.v))
        return sh, st, v


print('Функции готовы.')

## 3. Функция движения одного поезда

`simulate_train` считает движение одного поезда. Параметр `leading` — траектория переднего
поезда (или `None`, если это первый).

### Как работает выбег

Для выбега запоминаем две вещи:
- `last_powered` — последний режим с тягой или тормозом ('разгон' или 'торможение')
- `time_in_neutral` — сколько секунд уже идёт без тяги и без тормоза

Правила:
1. Если пора тормозить, но мы только что разгонялись и накопили меньше `COAST_MIN` секунд
   без тяги — сначала ставим выбег (a = 0), а не тормоз.
2. То же при обратном переходе: после торможения нельзя сразу включать тягу — сначала выбег.
3. Чтобы выбег включался заранее, условие тормоза расширили:
   расстояние до точки тормоза ≤ тормозной путь + `v · COAST_MIN` + запас.
   Тогда тяга снимается раньше, и за время выбега как раз накапливается ≥ 30 с.

### Учёт переднего поезда

На каждом шаге смотрим, где сейчас находится передний поезд. Опасность только если
**мы быстрее него и идём вдогон**. Тогда тормозим до его скорости, целясь в точку
`хвост_переднего − MIN_DIST_TO_LEAD`. Если расстояние уже меньше `3 · MIN_DIST_TO_LEAD`,
дополнительно ограничиваем скорость скоростью переднего, чтобы не приближаться.

In [ ]:
def simulate_train(train_id, t_start, leading=None, max_steps=100000):
    """Считаем движение одного поезда.

    train_id — номер поезда.
    t_start  — абсолютное время начала движения, с.
    leading  — траектория переднего поезда (None для первого).
    Возвращает DataFrame с состоянием на каждом шаге.
    """
    s_head = 0.0
    s_tail = -L_train
    v      = 0.0
    t_abs  = float(t_start)

    last_powered    = None    # None / 'разгон' / 'торможение'
    time_in_neutral = 0.0

    results = []
    step = 0

    while step < max_steps:
        # 0. Эффективная точка остановки.
        # Если передний поезд уже стоит, наша конечная сдвигается на MIN_DIST_TO_LEAD назад.
        if leading is not None and t_abs >= leading.t_max:
            s_stop_eff = min(S_stop, leading.s_t_final - MIN_DIST_TO_LEAD)
        else:
            s_stop_eff = S_stop

        if s_head >= s_stop_eff:
            break

        # 1. Текущие ограничения скорости (по голове и хвосту)
        v_lim_head  = get_speed_limit(s_head, speed_limits_ms)
        v_lim_tail  = get_speed_limit_for_tail(max(s_tail, 0), speed_limits_ms)
        v_limit_eff = min(v_lim_head, v_lim_tail)

        # 2. Ближайшая статическая точка снижения скорости
        dist_static, v_tgt_static = find_next_static_restriction(
            s_head, v, speed_limits_ms, s_stop_eff
        )

        # 3. Ограничение от переднего поезда
        dyn_active = False
        active_restriction = 'статическое'
        if leading is not None:
            s_h_lead, s_t_lead, v_lead = leading.at(t_abs)
            D = s_t_lead - s_head
            if D > 0 and v > v_lead:
                # догоняем — нужен тормозной путь до safe-точки
                dist_dyn   = max(D - MIN_DIST_TO_LEAD, 0.0)
                v_tgt_dyn  = v_lead
                dyn_active = True
            # близко к переднему — не разгоняемся выше его скорости
            if D > 0 and D < 3 * MIN_DIST_TO_LEAD:
                v_limit_eff = min(v_limit_eff, v_lead)

        # 4. Берём более строгое ограничение: статика или передний поезд
        s_brake_static = braking_distance_to_speed(v, v_tgt_static, a_brake)
        if dyn_active:
            s_brake_dyn = braking_distance_to_speed(v, v_tgt_dyn, a_brake)
            slack_static = dist_static - s_brake_static
            slack_dyn    = dist_dyn    - s_brake_dyn
            if slack_dyn < slack_static:
                dist_to_target, v_target, s_brake_needed = dist_dyn, v_tgt_dyn, s_brake_dyn
                active_restriction = f'поезд №{train_id - 1}'
            else:
                dist_to_target, v_target, s_brake_needed = dist_static, v_tgt_static, s_brake_static
        else:
            dist_to_target, v_target, s_brake_needed = dist_static, v_tgt_static, s_brake_static

        # 5. Когда включать тормоз: заранее (через выбег) или сразу
        need_brake_prep = dist_to_target <= s_brake_needed + v * COAST_MIN + safety_margin
        need_brake_now  = dist_to_target <= s_brake_needed + safety_margin

        # 6. Выбор режима
        if need_brake_now:
            # тормозить надо уже сейчас
            if last_powered == 'разгон' and time_in_neutral < COAST_MIN:
                mode, a = 'выбег', 0.0     # принудительный выбег перед тормозом
            else:
                mode, a = 'торможение', a_brake
        elif need_brake_prep:
            # тормозить скоро — пора снять тягу
            if last_powered == 'разгон':
                mode, a = 'выбег', 0.0
            else:
                mode, a = 'равномерно', 0.0
        elif v >= v_limit_eff - 0.01:
            mode, a = 'равномерно', 0.0
        elif last_powered == 'торможение' and time_in_neutral < COAST_MIN:
            # только что тормозили — нельзя сразу включать тягу
            mode, a = 'выбег', 0.0
        else:
            mode, a = 'разгон', a_acc

        # 7. Новая скорость
        v_new = speed_after_step(v, a, ds)
        if v_new > v_limit_eff:
            v_new = v_limit_eff
        if v_new < 0:
            v_new = 0.0

        # 8. Время на прохождение шага
        v_avg = (v + v_new) / 2
        if v_avg > 0.01:
            dt_step = ds / v_avg
        else:
            if a > 0:
                dt_step = np.sqrt(2 * ds / a)
                v_new   = a * dt_step
            else:
                # стоим и не разгоняемся — выходим
                break

        # 9. Обновляем счётчики выбега
        if mode == 'разгон':
            last_powered, time_in_neutral = 'разгон', 0.0
        elif mode == 'торможение':
            last_powered, time_in_neutral = 'торможение', 0.0
        else:
            time_in_neutral += dt_step

        # 10. Запись шага
        results.append({
            'поезд':              train_id,
            'шаг':                step,
            't_абс_с':            t_abs,
            't_от_старта_с':      t_abs - t_start,
            's_голова_м':         s_head,
            's_хвост_м':          s_tail,
            'v_м/с':              v,
            'v_км/ч':             v * 3.6,
            'a_м/с²':             a,
            'режим':              mode,
            'v_огр_эфф_км/ч':     v_limit_eff * 3.6,
            'тормозной_путь_м':   s_brake_needed,
            'расст_до_цели_м':    dist_to_target,
            'v_цель_км/ч':        v_target * 3.6,
            'активное_огр':       active_restriction,
            't_в_нейтрали_с':     time_in_neutral,
            'dt_шага_с':          dt_step,
        })

        # 11. Сдвиг
        t_abs  += dt_step
        s_head += ds
        s_tail  = s_head - L_train
        v       = v_new
        step   += 1

    return pd.DataFrame(results)


print('Функция simulate_train готова.')

## 4. Прогон группы

Считаем поезда по очереди. Каждому следующему передаём траекторию предыдущего.

In [ ]:
all_dfs = []
leading_traj = None

for train_id in range(1, N_TRAINS + 1):
    t_start = (train_id - 1) * HEADWAY_SEC
    print(f'>>> Поезд №{train_id}: старт в t = {t_start:.0f} с ({t_start/60:.1f} мин)')
    df_i = simulate_train(train_id, t_start, leading=leading_traj)
    t_end = df_i['t_абс_с'].iloc[-1]
    travel = t_end - t_start
    print(f'    шагов: {len(df_i)}, в пути: {travel:.0f} с ({travel/60:.1f} мин), '
          f'конечная s головы: {df_i["s_голова_м"].iloc[-1]:.0f} м')
    all_dfs.append(df_i)
    leading_traj = LeadingTrajectory(df_i)

# Сводим всё в один DataFrame
df_all = pd.concat(all_dfs, ignore_index=True)
print(f'\nВсего записей: {len(df_all)}')
df_all.head()

## 5. Сводка по поездам

In [ ]:
summary = []
for tid, df_i in enumerate(all_dfs, 1):
    t_start = df_i['t_абс_с'].iloc[0]
    t_end   = df_i['t_абс_с'].iloc[-1]
    travel  = t_end - t_start
    s_end   = df_i['s_голова_м'].iloc[-1]
    v_max   = df_i['v_км/ч'].max()
    v_mean  = (s_end / travel) * 3.6 if travel > 0 else 0
    summary.append({
        'поезд':            tid,
        'старт_мин':        t_start / 60,
        'финиш_мин':        t_end / 60,
        'в_пути_мин':       travel / 60,
        'пройдено_км':      s_end / 1000,
        'v_max_км/ч':       v_max,
        'v_средняя_км/ч':   v_mean,
        'шагов_всего':      len(df_i),
    })

df_summary = pd.DataFrame(summary)
print('Сводка по группе:')
df_summary

In [ ]:
# Сколько шагов в каждом режиме у каждого поезда
mode_dist = df_all.groupby(['поезд', 'режим']).size().unstack(fill_value=0)
mode_dist['всего'] = mode_dist.sum(axis=1)
print('Шагов по режимам:')
mode_dist

In [ ]:
# Проверка длительности выбегов — должна быть от 30 до 60 с
coast_blocks = []
for tid, df_i in enumerate(all_dfs, 1):
    # разбиваем шаги на блоки одного режима подряд
    blocks = (df_i['режим'] != df_i['режим'].shift(1)).cumsum()
    for block_id, g in df_i.groupby(blocks):
        if g['режим'].iloc[0] == 'выбег':
            duration = g['t_абс_с'].iloc[-1] - g['t_абс_с'].iloc[0] + g['dt_шага_с'].iloc[-1]
            coast_blocks.append({
                'поезд':     tid,
                'начало_с':  g['t_от_старта_с'].iloc[0],
                'начало_м':  g['s_голова_м'].iloc[0],
                'длит_с':    duration,
                'длит_м':    g['s_голова_м'].iloc[-1] - g['s_голова_м'].iloc[0] + ds,
            })
df_coast = pd.DataFrame(coast_blocks)
print(f'Всего эпизодов выбега: {len(df_coast)}')
print(f'Длительность, с — min={df_coast["длит_с"].min():.1f}, '
      f'max={df_coast["длит_с"].max():.1f}, среднее={df_coast["длит_с"].mean():.1f}')
print('(требование: 30–60 с)')
df_coast

## 6. Проверка дистанций

Для каждой пары соседних поездов считаем минимальное расстояние от головы догоняющего
до хвоста переднего. Должно быть не меньше `MIN_DIST_TO_LEAD`.

In [ ]:
for i in range(1, N_TRAINS):
    df_lead = all_dfs[i - 1]
    df_foll = all_dfs[i]
    lt = LeadingTrajectory(df_lead)
    min_d = float('inf')
    min_t = None
    for _, row in df_foll.iterrows():
        _, s_t_lead, v_lead = lt.at(row['t_абс_с'])
        # учитываем только когда уже отъехали от старта
        # (на старте оба в одной точке, но передний уже уходит)
        if row['v_м/с'] > v_lead or row['s_голова_м'] > 100:
            d = s_t_lead - row['s_голова_м']
            if d < min_d:
                min_d = d
                min_t = row['t_абс_с']
    print(f'Поезда {i} → {i+1}: минимальная дистанция = {min_d:.0f} м '
          f'(в момент t = {min_t:.0f} с)')
    if min_d < MIN_DIST_TO_LEAD:
        print(f'   нарушение (надо {MIN_DIST_TO_LEAD} м)')
    else:
        print(f'   ок, дистанция выдержана')

## 7. Графики

### 7.1. Скорость по пути

Профили скорости трёх поездов в основном совпадают — поезда одинаковые. Различия там,
где догоняющий подстраивается под скорость переднего.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
for tid, df_i in enumerate(all_dfs, 1):
    ax.plot(df_i['s_голова_м'] / 1000, df_i['v_км/ч'],
            color=colors[tid - 1], linewidth=1.6, label=f'Поезд №{tid}')
# линия ограничений (для головы)
lim_x, lim_y = [], []
for s1, s2, v in speed_limits_kmh:
    lim_x += [s1 / 1000, s2 / 1000]
    lim_y += [v, v]
ax.plot(lim_x, lim_y, 'r--', linewidth=1, alpha=0.6, label='Ограничение скорости')
# фон в зонах пониженной скорости
for s1, s2, v in speed_limits_kmh:
    if v < 80:
        ax.axvspan(s1 / 1000, s2 / 1000, color='red', alpha=0.05)
ax.set_xlabel('Путь (голова поезда), км')
ax.set_ylabel('Скорость, км/ч')
ax.set_title('Скорость поездов по пути')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')
ax.set_ylim(0, 95)
plt.tight_layout()
plt.show()

### 7.2. График «путь – время»

По оси X — время, по оси Y — путь. Сплошная линия — голова поезда, штриховая — хвост.
Линии разных поездов не пересекаются — значит, столкновений нет.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for tid, df_i in enumerate(all_dfs, 1):
    c = colors[tid - 1]
    ax.plot(df_i['t_абс_с'] / 60, df_i['s_голова_м'] / 1000,
            color=c, linewidth=1.8, label=f'Поезд №{tid} — голова')
    ax.plot(df_i['t_абс_с'] / 60, df_i['s_хвост_м'] / 1000,
            color=c, linewidth=1, linestyle='--', alpha=0.6,
            label=f'Поезд №{tid} — хвост')
# фон в зонах пониженной скорости
for s1, s2, v in speed_limits_kmh:
    if v < 80:
        ax.axhspan(s1 / 1000, s2 / 1000, color='red', alpha=0.05)
ax.set_xlabel('Время, мин')
ax.set_ylabel('Путь, км')
ax.set_title('Путь – время для группы поездов')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(-1.2, 31)
plt.tight_layout()
plt.show()

### 7.3. Режимы движения

Шаги каждого поезда раскрашены по режиму. Видно, что выбег (серый) всегда между
разгоном (зелёный) и торможением (красный).

In [ ]:
mode_colors = {
    'разгон':     '#2ca02c',  # зелёный
    'равномерно': '#1f77b4',  # синий
    'выбег':      '#7f7f7f',  # серый
    'торможение': '#d62728',  # красный
}

fig, axes = plt.subplots(N_TRAINS, 1, figsize=(14, 2.5 * N_TRAINS), sharex=True)
if N_TRAINS == 1:
    axes = [axes]
for tid, df_i in enumerate(all_dfs, 1):
    ax = axes[tid - 1]
    for mode, c in mode_colors.items():
        m = df_i['режим'] == mode
        ax.scatter(df_i.loc[m, 's_голова_м'] / 1000, df_i.loc[m, 'v_км/ч'],
                   c=c, s=10, label=mode)
    ax.plot(lim_x, lim_y, 'r--', linewidth=1, alpha=0.5)
    ax.set_ylabel('v, км/ч')
    ax.set_title(f'Поезд №{tid} — режимы')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 95)
    if tid == 1:
        ax.legend(loc='lower right', ncol=4)
axes[-1].set_xlabel('Путь (голова поезда), км')
plt.tight_layout()
plt.show()

### 7.4. Дистанция между поездами

На каждый момент времени берём положение хвоста переднего и головы догоняющего —
это и есть расстояние между ними.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for i in range(1, N_TRAINS):
    df_lead = all_dfs[i - 1]
    df_foll = all_dfs[i]
    lt = LeadingTrajectory(df_lead)
    t_arr   = df_foll['t_абс_с'].to_numpy()
    s_head_foll = df_foll['s_голова_м'].to_numpy()
    dist = np.array([lt.at(t)[1] - s for t, s in zip(t_arr, s_head_foll)])
    ax.plot(t_arr / 60, dist, label=f'Поезд {i+1} ← {i}', linewidth=1.5)
ax.axhline(MIN_DIST_TO_LEAD, color='red', linestyle='--', alpha=0.7,
           label=f'Мин. дистанция ({MIN_DIST_TO_LEAD} м)')
ax.set_xlabel('Время, мин')
ax.set_ylabel('Расстояние «голова догоняющего → хвост переднего», м')
ax.set_title('Дистанция между соседними поездами')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### 7.5. Скорость по времени

На графике 7.1 («скорость по пути») кривые трёх поездов накладываются, потому что поезда
одинаковые и едут по одному маршруту — на одной и той же координате у них и скорость
одинаковая. Если построить скорость по **времени**, поезда сдвинутся друг относительно
друга на интервал отправления (10 минут).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for tid, df_i in enumerate(all_dfs, 1):
    ax.plot(df_i['t_абс_с'] / 60, df_i['v_км/ч'],
            color=colors[tid - 1], linewidth=1.6, label=f'Поезд №{tid}')
ax.set_xlabel('Время, мин')
ax.set_ylabel('Скорость, км/ч')
ax.set_title('Скорость поездов по времени')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')
ax.set_ylim(0, 95)
plt.tight_layout()
plt.show()

## 8. Сохранение данных

Все данные по всем поездам лежат в `df_all`. Сохраним в CSV для дальнейшей работы.

In [ ]:
df_all.to_csv('movement_log.csv', index=False, encoding='utf-8-sig')
print(f'Сохранено: movement_log.csv, {len(df_all)} строк, {len(df_all.columns)} столбцов')
print('Столбцы:', list(df_all.columns))

## 9. Выводы

1. Движение одного поезда вынесено в функцию `simulate_train(train_id, t_start, leading)`.
2. Поезда считаются по очереди: каждый следующий получает траекторию предыдущего.
3. Выбег сделан через два счётчика (`last_powered`, `time_in_neutral`).
   Условие тормоза расширили на `v · COAST_MIN`, чтобы тяга снималась заранее
   и за время выбега успевало накопиться не меньше 30 с.
4. Передний поезд — это подвижное ограничение: тормозим в точку `хвост − MIN_DIST_TO_LEAD`
   до его скорости. Вблизи дополнительно не разгоняемся выше его скорости.
5. Все шаги всех поездов лежат в общем `df_all`. По нему легко смотреть статистику
   по режимам, выбегам и дистанциям.

Параметры (`N_TRAINS`, `HEADWAY_SEC`, `MIN_DIST_TO_LEAD`, `COAST_MIN`) меняются в одной
ячейке в начале — поведение всей группы сразу пересчитывается.